In [1]:
#Task 1: Load Spotify song data and split into X and y
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sp = pd.read_csv("spotify_popularity.csv")
X = sp.drop(columns="popularity")
y = sp["popularity"]

# scale the features so that the penalty treats all of them equally
sc = StandardScaler()
Xs = pd.DataFrame(sc.fit_transform(X), columns=X.columns)

print("Features:", list(X.columns))
print("X shape:", Xs.shape, "y shape:", y.shape)

Features: ['danceability', 'energy', 'loudness', 'valence', 'acousticness', 'tempo', 'duration_min', 'key']
X shape: (150, 8) y shape: (150,)


In [2]:
#Task 2: Lasso regression coefficients
from sklearn.linear_model import Lasso, Ridge, ElasticNet

las = Lasso(alpha=0.7)
las.fit(Xs, y)

for i in range(len(X.columns)):
    print(X.columns[i], "->", round(las.coef_[i], 3) + 0)

danceability -> 7.244
energy -> 3.715
loudness -> 3.009
valence -> 0.0
acousticness -> -1.937
tempo -> 0.0
duration_min -> 0.0
key -> 0.0


In [3]:
#Task 3: Ridge regression and comparison with Lasso
rid = Ridge(alpha=0.7)
rid.fit(Xs, y)

comp = pd.DataFrame({"Lasso": las.coef_, "Ridge": rid.coef_}, index=X.columns).round(3) + 0
print(comp)

# Lasso sets valence, tempo, duration_min and key exactly to 0, so it removes those features.
# Ridge keeps all of them as small non-zero values (for example duration_min is about 0.66), it only shrinks the coefficients.
# Both agree that danceability, energy, loudness and acousticness are the important features.

              Lasso  Ridge
danceability  7.244  8.130
energy        3.715  4.537
loudness      3.009  3.667
valence       0.000 -0.118
acousticness -1.937 -2.677
tempo         0.000  0.294
duration_min  0.000  0.658
key           0.000 -0.196


In [4]:
#Task 4: ElasticNet with different l1_ratio values
res = {"Lasso": las.coef_, "Ridge": rid.coef_}
for r in [0.2, 0.5, 0.8]:
    en = ElasticNet(alpha=0.7, l1_ratio=r)
    en.fit(Xs, y)
    res["EN l1=" + str(r)] = en.coef_

print(pd.DataFrame(res, index=X.columns).round(3) + 0)

# With l1_ratio 0.2 ElasticNet behaves more like Ridge: valence and tempo stay as small non-zero values.
# As l1_ratio goes up to 0.8 it behaves more like Lasso: tempo becomes 0 and valence is almost 0.
# The main features (danceability, energy, loudness) are a bit smaller than in Lasso because the ridge part also shrinks them.

              Lasso  Ridge  EN l1=0.2  EN l1=0.5  EN l1=0.8
danceability  7.244  8.130      4.963      5.602      6.466
energy        3.715  4.537      2.632      2.927      3.342
loudness      3.009  3.667      2.222      2.438      2.750
valence       0.000 -0.118     -0.408     -0.256     -0.006
acousticness -1.937 -2.677     -1.438     -1.552     -1.762
tempo         0.000  0.294     -0.104      0.000      0.000
duration_min  0.000  0.658      0.000      0.000      0.000
key           0.000 -0.196      0.000      0.000      0.000


In [5]:
#Task 5: Feature selection with Lasso for a Zomato-style rating predictor
zo = pd.read_csv("zomato_rating_data.csv")
Xz = zo.drop(columns="rating")
yz = zo["rating"]
Xzs = pd.DataFrame(StandardScaler().fit_transform(Xz), columns=Xz.columns)

sel = Lasso(alpha=0.05)
sel.fit(Xzs, yz)

kept = []
dropped = []
for i in range(len(Xz.columns)):
    print(Xz.columns[i], "->", round(sel.coef_[i], 4) + 0)
    if sel.coef_[i] != 0:
        kept.append(Xz.columns[i])
    else:
        dropped.append(Xz.columns[i])

print("\nKept features:", kept)
print("Dropped features:", dropped)

votes -> 0.2284
average_cost_for_two -> 0.1578
online_order -> 0.0
table_booking -> 0.0417
delivery_time -> -0.0695
cuisines_count -> 0.0
distance_km -> 0.0

Kept features: ['votes', 'average_cost_for_two', 'table_booking', 'delivery_time']
Dropped features: ['online_order', 'cuisines_count', 'distance_km']
